### Meng-import Library

In [75]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [76]:
df = pd.read_csv("ecommerce-sales-data -2019.csv")

In [77]:
df.head()

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,Postal Code
0,176558,USB-C Charging Cable,2,11.95,04/19/19 08:46,917 1st St,Dallas,TX 75001
1,176559,Bose SoundSport Headphones,1,99.99,04/07/2019 22:30,682 Chestnut St,Boston,MA 02215
2,176560,Google Phone,1,600,04/12/2019 14:38,669 Spruce St,Los Angeles,CA 90001
3,176560,Wired Headphones,1,11.99,04/12/2019 14:38,669 Spruce St,Los Angeles,CA 90001
4,176561,Wired Headphones,1,11.99,04/30/19 09:27,333 8th St,Los Angeles,CA 90001


### Memeriksa Tipe Data Tiap Kolom

In [78]:
df.dtypes

Order ID            object
Product             object
Quantity Ordered    object
Price Each          object
Order Date          object
Purchase Address    object
City                object
Postal Code         object
dtype: object

### Membuat Fungsi untuk Memeriksa Nilai Kosong Tiap Kolom

In [79]:
def fungsi_hitung_null(df):

    jumlah_null = df.isna().sum()
    presentase_null = (jumlah_null/len(df))*100

    tabel_null = pd.DataFrame({"Jumlah Null" : jumlah_null, "Presentase Null" : presentase_null})
    return tabel_null

In [80]:
fungsi_hitung_null(df)

,Jumlah Null,Presentase Null
Order ID,0,0.000000
Product,0,0.000000
Quantity Ordered,0,0.000000
Price Each,0,0.000000
Order Date,0,0.000000
Purchase Address,0,0.000000
City,0,0.000000
Postal Code,7,0.181253


### Menentukan Anomali Input Data di Kolom "Order ID"
Ditemukan beberapa baris dalam tabel mengandung informasi yang tidak sesuai. Tidak ada informasi yang bisa didapatkan dari beberapa baris tersebut, sepertinya terjadi salah ketik saat memasukan data, sehingga langkah yang akan diambil adalah menghapus 10 baris tersebut dari tabel.

In [81]:
bukan_angka = df[pd.to_numeric(df['Order ID'], errors='coerce').isna()]
bukan_angka

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,Postal Code
75,Order ID,Product,Quantity Ordered,Price Each,Order Date,Address,City,Zip Code
566,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,NaN
598,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,NaN
921,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,NaN
2027,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,NaN
2641,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,NaN
3072,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,NaN
3138,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,NaN
3787,Order ID,Product,Quantity Ordered,Price Each,Order Date,Address,City,Zip Code
3840,Order ID,Product,Quantity Ordered,Price Each,Order Date,Address,City,Zip Code


In [82]:
df_clean = df[pd.to_numeric(df['Order ID'], errors='coerce').notna()].copy()
df_clean['Order ID'] = df_clean['Order ID'].astype('string') # mengubah tipe data menjadi string
df_clean.dtypes

Order ID            string[python]
Product                     object
Quantity Ordered            object
Price Each                  object
Order Date                  object
Purchase Address            object
City                        object
Postal Code                 object
dtype: object

Memeriksa apakah terdapat data di Order ID yang tidak valid (berisi kurang dari atau lebih dari 6 digit angka).

In [83]:
bukan_6_digit = df_clean[df_clean['Order ID'].str.len() != 6]
bukan_6_digit

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,Postal Code


### Menentukan Anomali Data di Kolom "Product"

In [84]:
df_clean["Product"].nunique()

19

Memeriksa apakah ada salah ketik dalam penulisan Produk.

In [85]:
df_clean["Product"].nunique()

19

In [86]:
df_clean["Product"].unique()

array(['USB-C Charging Cable', 'Bose SoundSport Headphones',
       'Google Phone', 'Wired Headphones', 'Macbook Pro Laptop',
       'Lightning Charging Cable', '27in 4K Gaming Monitor',
       'AA Batteries (4-pack)', 'Apple Airpods Headphones',
       'AAA Batteries (4-pack)', 'iPhone', 'Flatscreen TV',
       '27in FHD Monitor', '20in Monitor', 'LG Dryer', 'ThinkPad Laptop',
       'Vareebadd Phone', 'LG Washing Machine', '34in Ultrawide Monitor'],
      dtype=object)

In [87]:
df_clean["Product"] = df_clean["Product"].astype('category') # mengubah tipe data object menjadi category untuk menghemat memori RAM

In [88]:
df_clean.dtypes

Order ID            string[python]
Product                   category
Quantity Ordered            object
Price Each                  object
Order Date                  object
Purchase Address            object
City                        object
Postal Code                 object
dtype: object

### Menentukan Anomali Data dalam Kolom "Quantity Ordered"
Memeriksa apakah terdapat nilai string di dalam kolom.

In [89]:
bukan_angka = df_clean[pd.to_numeric(df_clean['Quantity Ordered'], errors='coerce').isna()]
bukan_angka

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,Postal Code


Memeriksa apakah terdapat nilai float dalam kolom kuantitas yang seharusnya bilangan bulat.

In [90]:
ada_float = df_clean[df_clean['Quantity Ordered'].astype(str).str.contains(r'\d+\.\d+', na=False)]
ada_float

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,Postal Code


In [91]:
df_clean['Quantity Ordered'] = df_clean['Quantity Ordered'].astype(int)
df_clean.dtypes

Order ID            string[python]
Product                   category
Quantity Ordered             int32
Price Each                  object
Order Date                  object
Purchase Address            object
City                        object
Postal Code                 object
dtype: object

### Menentukan Anomali Data dalam Kolom "Price Each"

Terdapat 17 harga berbeda untuk 19 kategori produk berbeda maka ada kemungkinan 3 produk dengan harga serupa, tapi perlu diperiksa kembali.

In [92]:
df_clean["Price Each"].nunique()

17

In [93]:
df_clean["Price Each"].unique() # semuanya angka dan tidak ditemukan string

array(['11.95', '99.99', '600', '11.99', '1700', '14.95', '389.99',
       '3.84', '150', '2.99', '700', '300', '149.99', '109.99', '999.99',
       '400', '379.99'], dtype=object)

In [94]:
df_clean["Price Each"] = df_clean["Price Each"].astype(float)

In [95]:
produk_modus_harga = df_clean.groupby('Product')['Price Each'].agg(lambda x: x.mode()[0]).reset_index()
produk_modus_harga.columns = ['Product', 'Harga_Sebenarnya']
produk_modus_harga.sort_values(by="Harga_Sebenarnya")

C:\Users\anggi\AppData\Local\Temp\ipykernel_20896\569547424.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  produk_modus_harga = df_clean.groupby('Product')['Price Each'].agg(lambda x: x.mode()[0]).reset_index()


,Product,Harga_Sebenarnya
5,AAA Batteries (4-pack),2.99
4,AA Batteries (4-pack),3.84
15,USB-C Charging Cable,11.95
17,Wired Headphones,11.99
12,Lightning Charging Cable,14.95
7,Bose SoundSport Headphones,99.99
0,20in Monitor,109.99
2,27in FHD Monitor,149.99
6,Apple Airpods Headphones,150.00
8,Flatscreen TV,300.00


In [96]:
cek_harga = (df_clean.groupby('Price Each')['Product'].nunique().reset_index(name='Jumlah_Produk'))

harga_dan_produk = cek_harga[cek_harga['Jumlah_Produk'] >= 1]

harga_dan_produk

,Price Each,Jumlah_Produk
0,2.99,1
1,3.84,1
2,11.95,1
3,11.99,1
4,14.95,1
5,99.99,1
6,109.99,1
7,149.99,1
8,150.00,1
9,300.00,1


In [97]:
harga_dan_produk["Jumlah_Produk"].sum() 

19

In [98]:
df_clean["Product"].nunique()

19

Hasilnya sesuai dengan banyaknya kategori produk yakni 19 maka tidak terdapat kesalahan input pada harga.

### Menentukan Ketidaksesuaian Format Tanggal dalam Kolom "Order Date"

In [99]:
df_clean.dtypes

Order ID            string[python]
Product                   category
Quantity Ordered             int32
Price Each                 float64
Order Date                  object
Purchase Address            object
City                        object
Postal Code                 object
dtype: object

Kolom Order Date memiliki ketidakkonsistenan format tahun, yaitu menggunakan YYYY dan YY. Namun struktur tanggal tetap menggunakan format MM/DD/YY(YY) HH:MM.

In [100]:
df_clean['Order Date'].sample(10)

2217      03/19/19 21:10
432     08/09/2019 20:21
1817      07/28/19 18:08
1464    01/03/2019 21:53
1978    06/12/2019 13:42
3451      10/18/19 11:36
2196      06/19/19 18:03
749       12/20/19 18:19
1798      07/31/19 10:59
2964    03/12/2019 18:33
Name: Order Date, dtype: object

Memeriksa apakah ada tanggal yang tidak bisa dokonfersi ke format tanggal.

In [101]:
tanggal = pd.to_datetime(df_clean['Order Date'],format='mixed',errors='coerce')
tanggal_bermasalah = df_clean[tanggal.isna()]

tanggal_bermasalah

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,Postal Code


In [102]:
df_clean["Order Date"] = pd.to_datetime(df_clean["Order Date"], format="mixed")

In [103]:
df_clean.dtypes

Order ID            string[python]
Product                   category
Quantity Ordered             int32
Price Each                 float64
Order Date          datetime64[ns]
Purchase Address            object
City                        object
Postal Code                 object
dtype: object

In [104]:
fungsi_hitung_null(df_clean)

,Jumlah Null,Presentase Null
Order ID,0,0.0
Product,0,0.0
Quantity Ordered,0,0.0
Price Each,0,0.0
Order Date,0,0.0
Purchase Address,0,0.0
City,0,0.0
Postal Code,0,0.0


### Menentukan Anomali pada Kolom "Purchase Address"

In [105]:
df_clean.dtypes

Order ID            string[python]
Product                   category
Quantity Ordered             int32
Price Each                 float64
Order Date          datetime64[ns]
Purchase Address            object
City                        object
Postal Code                 object
dtype: object

In [106]:
df_clean["Purchase Address"].nunique()

3579

In [107]:
df_clean.head()

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,Postal Code
0,176558,USB-C Charging Cable,2,11.95,2019-04-19 08:46:00,917 1st St,Dallas,TX 75001
1,176559,Bose SoundSport Headphones,1,99.99,2019-04-07 22:30:00,682 Chestnut St,Boston,MA 02215
2,176560,Google Phone,1,600.00,2019-04-12 14:38:00,669 Spruce St,Los Angeles,CA 90001
3,176560,Wired Headphones,1,11.99,2019-04-12 14:38:00,669 Spruce St,Los Angeles,CA 90001
4,176561,Wired Headphones,1,11.99,2019-04-30 09:27:00,333 8th St,Los Angeles,CA 90001


In [108]:
df_clean['Purchase Address'].astype('string').str.isnumeric()

0       False
1       False
2       False
3       False
4       False
        ...  
3857    False
3858    False
3859    False
3860    False
3861    False
Name: Purchase Address, Length: 3852, dtype: boolean

Tidak ditemukan data pada kolom yang hanya berisi angka.

In [109]:
df_clean[df_clean["Purchase Address"].astype('string').str.isnumeric() == True]

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,Postal Code


In [113]:
df_clean["Purchase Address"] = df_clean["Purchase Address"].astype('string')

### Menentukan Anomali pada Kolom "City"

In [114]:
df_clean.dtypes

Order ID            string[python]
Product                   category
Quantity Ordered             int32
Price Each                 float64
Order Date          datetime64[ns]
Purchase Address    string[python]
City                        object
Postal Code                 object
dtype: object

Mencari nama kota yang terdiri dari gabungan angka dan huruf.

In [115]:
df_clean[df_clean['City'].str.contains(r'\d', na=False)]

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,Postal Code


In [117]:
df_clean["City"].nunique()

9

In [118]:
df_clean["City"].unique()

array([' Dallas', ' Boston', ' Los Angeles', ' San Francisco', ' Seattle',
       ' Atlanta', ' New York City', ' Portland', ' Austin'], dtype=object)

In [120]:
df_clean["City"] = df_clean["City"].astype('category')

### Menentukan Anomali pada Kolom "Postal Code"

In [134]:
df_clean.dtypes

Order ID            string[python]
Product                   category
Quantity Ordered             int32
Price Each                 float64
Order Date          datetime64[ns]
Purchase Address    string[python]
City                      category
Postal Code               category
dtype: object

In [122]:
df_clean["Postal Code"].nunique()

10

In [123]:
df_clean["Postal Code"].unique()

array([' TX 75001', ' MA 02215', ' CA 90001', ' CA 94016', ' WA 98101',
       ' GA 30301', ' NY 10001', ' OR 97035', ' TX 73301', ' ME 04101'],
      dtype=object)

In [133]:
df_clean["Postal Code"] = df_clean["Postal Code"].astype('category')

Memeriksa apakah ada postal code yang digunakan untuk lebih dari satu kota.

In [ ]:
cek_postal = (
    df_clean.groupby('Postal Code')['City']
    .nunique()
    .reset_index(name='Jumlah_City')
)

cek_postal

C:\Users\anggi\AppData\Local\Temp\ipykernel_20896\2494125319.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_clean.groupby('Postal Code')['City']


,Postal Code,Jumlah_City
0,CA 90001,1
1,CA 94016,1
2,GA 30301,1
3,MA 02215,1
4,ME 04101,1
5,NY 10001,1
6,OR 97035,1
7,TX 73301,1
8,TX 75001,1
9,WA 98101,1


### Mengupload Data Menjadi File CSV

In [142]:
df_clean.dtypes # semua kolom sudah memiliki tipe data yang sesuai.

Order ID            string[python]
Product                   category
Quantity Ordered             int32
Price Each                 float64
Order Date          datetime64[ns]
Purchase Address    string[python]
City                      category
Postal Code               category
dtype: object

In [143]:
df_clean.head()

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,Postal Code
0,176558,USB-C Charging Cable,2,11.95,2019-04-19 08:46:00,917 1st St,Dallas,TX 75001
1,176559,Bose SoundSport Headphones,1,99.99,2019-04-07 22:30:00,682 Chestnut St,Boston,MA 02215
2,176560,Google Phone,1,600.00,2019-04-12 14:38:00,669 Spruce St,Los Angeles,CA 90001
3,176560,Wired Headphones,1,11.99,2019-04-12 14:38:00,669 Spruce St,Los Angeles,CA 90001
4,176561,Wired Headphones,1,11.99,2019-04-30 09:27:00,333 8th St,Los Angeles,CA 90001


In [144]:
df_clean.tail()

,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,City,Postal Code
3857,248336,Apple Airpods Headphones,1,150.00,2019-09-27 19:52:00,47 Willow St,Atlanta,GA 30301
3858,248337,AA Batteries (4-pack),1,3.84,2019-09-08 19:04:00,739 Cedar St,Portland,OR 97035
3859,248338,USB-C Charging Cable,1,11.95,2019-09-16 10:48:00,811 12th St,Boston,MA 02215
3860,248339,Bose SoundSport Headphones,1,99.99,2019-09-08 21:04:00,723 North St,San Francisco,CA 94016
3861,248340,AAA Batteries (4-pack),1,2.99,2019-09-13 20:08:00,896 12th St,Seattle,WA 98101


In [146]:
fungsi_hitung_null(df_clean) # sudah tidak terdapat nilai null.

,Jumlah Null,Presentase Null
Order ID,0,0.0
Product,0,0.0
Quantity Ordered,0,0.0
Price Each,0,0.0
Order Date,0,0.0
Purchase Address,0,0.0
City,0,0.0
Postal Code,0,0.0


In [147]:
df_clean.to_csv('ecommerce-sales-clean.csv', index=False)